In [1]:
# step 1 - 라이브러리 불러오기

In [2]:
import FinanceDataReader as fdr
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import unicodedata
import datetime

print('라이브러리 임포트 완료')

라이브러리 임포트 완료


In [3]:
# step 2 - KOSPI 전체 종목 리스트 조회 

In [4]:
market = 'KOSPI'
df_market = fdr.StockListing(market)

print(f'전체 종목 수 : {len(df_market)}개')
print(f'컬럼 목록 : {df_market.columns.tolist}') # 리스트형태로 

df_market.head()

전체 종목 수 : 948개
컬럼 목록 : <bound method IndexOpsMixin.tolist of Index(['Unnamed: 0', 'Code', 'ISU_CD', 'Name', 'Market', 'Dept', 'Close',
       'ChangeCode', 'Changes', 'ChagesRatio', 'Open', 'High', 'Low', 'Volume',
       'Amount', 'Marcap', 'Stocks', 'MarketId'],
      dtype='str')>


,Unnamed: 0,Code,ISU_CD,Name,Market,Dept,Close,ChangeCode,Changes,ChagesRatio,Open,High,Low,Volume,Amount,Marcap,Stocks,MarketId
0,0,005930,KR7005930003,삼성전자,KOSPI,NaN,285500,1,17000,6.33,284500,288500,280000,36031094,10278092379571,1669112542584000,5846278608,STK
1,1,000660,KR7000660001,SK하이닉스,KOSPI,NaN,1880000,1,194000,11.51,1833000,1949000,1826000,7433039,14004335299500,1339880446200000,712702365,STK
2,2,402340,KR7402340004,SK스퀘어,KOSPI,NaN,1187000,1,89000,8.11,1175000,1205000,1130000,1345499,1577030905000,156634604182000,131958386,STK
3,3,005935,KR7005931001,삼성전자우,KOSPI,NaN,194900,1,12200,6.68,193600,197900,190100,6586322,1281788839232,156382147464700,802371203,STK
4,4,005380,KR7005380001,현대차,KOSPI,NaN,646000,1,33000,5.38,631000,654000,610000,2569392,1624698473000,132273516836000,204757766,STK


In [5]:
# step 3 - 한글 종목명 정규화

In [6]:
# 정규화 전후 비교 예시
# s1 = '삼성전자'           # NFC (일반적인 한글 입력)
# s2 = '\u삼\u성\u전\u자'  # NFD (분해된 형태, 눈에는 같아 보임)

def normalize_str(s):
    return unicodedata.normalize('NFKC', s).strip()

# 전각 문자 예시 (실제로 종목명에서 발생할 수 있는 케이스)
test_cases = [
    ('㈜카카오', '정규화 전'),
    (normalize_str('㈜카카오'), '정규화 후'),
]

for text, label in test_cases:
    print(f'{label}: {text}')

# 실제 적용: df_market 종목명 전체 정규화
df_market['Name'] = df_market['Name'].apply(normalize_str)
print('\n✅ 종목명 정규화 완료')
df_market[['Code', 'Name', 'Marcap']].head()

정규화 전: ㈜카카오
정규화 후: (주)카카오

✅ 종목명 정규화 완료


,Code,Name,Marcap
0,005930,삼성전자,1669112542584000
1,000660,SK하이닉스,1339880446200000
2,402340,SK스퀘어,156634604182000
3,005935,삼성전자우,156382147464700
4,005380,현대차,132273516836000


In [7]:
# step 4 - 시가총액 TOP10 추출

In [8]:
top10 = df_market.nlargest(10, 'Marcap').iloc[::-1] # 반전 

# 시가총약 단위 변환 : 원 -> 조 (1조 --> 10^12)
top10_display = top10[['Name', 'Marcap']].copy()
top10_display['시가총액(조)'] = (top10_display['Marcap'] / 1e12).round(1)

top10_display[['Name', '시가총액(조)']].reset_index(drop=True)

,Name,시가총액(조)
0,기아,68.2
1,HD현대중공업,71.9
2,삼성물산,73.3
3,두산에너빌리티,82.0
4,LG에너지솔루션,109.5
5,현대차,132.3
6,삼성전자우,156.4
7,SK스퀘어,156.6
8,SK하이닉스,1339.9
9,삼성전자,1669.1


In [9]:
# step 5 - 시가총액 TOP10 수평막대그래프

In [10]:
fig_top10 = go.Figure(go.Bar(
    x=top10['Marcap'] / 1e12,
    y=top10['Name'],
    orientation='h', # 수평 막대그래프(가로)
    text=top10['Marcap'] / 1e12,
    texttemplate='%{text:1f}조', # 소수 첫째자리까지, + '조' 단위
    marker_color='steelblue' # 막대 색상
))

fig_top10.update_layout(
    title=f'{market} 시가총액 TOP 10',
    xaxis_title='시가총액 (조)',
    yaxis_title='종목명',
    bargap=0.15, # 막대 사이 간격 (0~1)
    height=450
)

fig_top10.show()

In [11]:
# step 6 - 개별 종목 주가 데이터 조회

In [12]:
# 종목명 -> 종목코드 변환
target_name = '삼성전자'
code = df_market.loc[df_market['Name'] == target_name, 'Code'].values[0]
print(f'{target_name} 종목코드 : {code}')

삼성전자 종목코드 : 005930


In [14]:
# 주가 데이터 조회
start_date = '2024-01-01'
end_date = datetime.datetime.now().strftime('%Y-%m-%d')

df_stock = fdr.DataReader(code, start_date, end_date)

print(f'조회기간 : {start_date} ~ {end_date}')
print(f'데이터 수 : {len(df_stock)}일')
df_stock.tail # 최근 5일

조회기간 : 2024-01-01 ~ 2026-05-11
데이터 수 : 572일


<bound method NDFrame.tail of               Open    High     Low   Close    Volume    Change
Date                                                          
2024-01-02   78200   79800   78200   79600  17142847  0.014013
2024-01-03   78500   78800   77000   77000  21753644 -0.032663
2024-01-04   76100   77300   76100   76600  15324439 -0.005195
2024-01-05   76700   77100   76400   76600  11304316  0.000000
2024-01-08   77000   77500   76400   76500  11088724 -0.001305
...            ...     ...     ...     ...       ...       ...
2026-05-04  228000  232500  224000  232500  32920816  0.054422
2026-05-06  254000  270000  251000  266000  53097996  0.144086
2026-05-07  272000  277000  260000  271500  41404687  0.020677
2026-05-08  260000  270000  260000  268500  25875880 -0.011050
2026-05-11  284500  288500  280000  285500  34321592  0.063315

[572 rows x 6 columns]>

In [15]:
# step 7 - 현재가 / 전일 대비 확인

In [16]:
current = df_stock['Close'].iloc[-1] # 최근 종가
prev = df_stock['Close'].iloc[-2] # 전일 종가
delta = current - prev # 전일 대비 변동폭
pct = delta / prev * 100 # 등락률 (%)

direction = '' if delta > 0 else ''

print(f'[ {target_name} ({code}) ]')
print(f'현재가 : {current:,}원')
print(f'전일대비 : {direction} {abs(delta):,}원 ({pct:+.2f}%)')

[ 삼성전자 (005930) ]
현재가 : 285,500원
전일대비 :  17,000원 (+6.33%)


In [18]:
# step 8 - 종가 라인 차트 (단일 종목)

In [20]:
fig_line = px.line(
    df_stock,
    y='Close',
    title=f'{target_name} 종가추이',
    labels={'Close':'종가 (원)', 'Date':'날짜'}
)

fig_line.update_layout(height=400)
fig_line.show()

In [21]:
# step 9 - 다중 종목 종가 비교

In [22]:
# 비교할 종목 목록
compare_names = ['삼성전자', 'SK하이닉스', 'LG에너지솔루션']

dfs = []
for name in compare_names:
    matched = df_market.loc[df_market['Name'] == name, 'Code'].values
    if len(matched) == 0:
        print(f'{name}: 종목 코드를 찾을 수 없습니다.')
        continue

    c = matched[0]
    df_temp = fdr.DataReader(c, start_date, end_date)

    if not df_temp.empty:
        #Close 열만 추출하고, 열 이름을 종목명으로 변경
        dfs.append(df_temp[['Close']].rename(columns={'Close':name}))
        print(f'{name}({c}) 데이터 로드 완료 ({len(df_temp)}일)')

# 수평 병합
merged_df = pd.concat(dfs, axis=1)
print(f'병합결과 : {merged_df.shape}')
merged_df.tail()

삼성전자(005930) 데이터 로드 완료 (572일)
SK하이닉스(000660) 데이터 로드 완료 (572일)
LG에너지솔루션(373220) 데이터 로드 완료 (572일)
병합결과 : (572, 3)


,삼성전자,SK하이닉스,LG에너지솔루션
Date,,,
2026-05-04,232500,1447000,472000
2026-05-06,266000,1601000,482000
2026-05-07,271500,1654000,483000
2026-05-08,268500,1686000,476500
2026-05-11,285500,1880000,468000


In [23]:
# 다중 종목 라인 차트 
fig_multi = px.line(
    merged_df,
    title='주요 종목 종가 비교',
    labels={'value': '종가 (원)', 'Date': '날짜', 'variable': '종목'}
)

fig_multi.update_layout(height=450, legend_title='종목')
fig_multi.show()